# Exploración Inicial de Datos

In [16]:
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
RAW_DIR  = os.path.join(BASE_DIR, '01_datos', 'raw')

TABLAS = {
    'Salud: Consumo SPA': 'r_salud_consumo.xlsx',
    'Salud: Ideación / Intento': 'r_salud_ideacion.xlsx',
    'Salud: Suicidio': 'r_salud_suicidio.xlsx',
    'Programas Deporte': 'r_programas_deporte.xlsx',
    'Habitabilidad / Accesibilidad': 'r_condiciones_habitabilidad_accesibilidad_2024.xlsx',
    'Centros Culturales': 'r_centros_culturales_ciudad_de_bogota.xlsx',
}

print(f'Total tablas: {len(TABLAS)}\n')

Total tablas: 6



In [17]:
KEYWORDS = ['año', 'year', 'fecha', 'date', 'periodo', 'anio', 'vigencia', 'tiempo', 'ano']

def extract_temporal_range(df):
    # 1. Buscamos por palabras clave
    for col in df.columns:
        col_lower = str(col).lower()
        if any(kw in col_lower for kw in KEYWORDS) or pd.api.types.is_datetime64_any_dtype(df[col]):
            try:
                if pd.api.types.is_datetime64_any_dtype(df[col]):
                    years = df[col].dt.year.dropna().astype(int)
                else:
                    serie = pd.to_numeric(df[col], errors='coerce').dropna()
                    if serie.between(1900, 2100).mean() > 0.3:
                        years = serie.astype(int)
                    else:
                        continue
                if not years.empty:
                    return col, int(years.min()), int(years.max())
            except:
                continue
                
    # 2. Búsqueda exhaustiva si fallan las palabras clave
    for col in df.columns:
        try:
             serie = pd.to_numeric(df[col], errors='coerce').dropna()
             if not serie.empty and serie.between(1900, 2100).mean() > 0.85:
                 return col, int(serie.min()), int(serie.max())
        except:
             pass
             
    return None, None, None

resumen = []

for nombre, archivo in TABLAS.items():
    ruta = os.path.join(RAW_DIR, archivo)
    if not os.path.exists(ruta):
        print(f'❌ {nombre}: Archivo no encontrado ({archivo})')
        continue

    df = pd.read_excel(ruta, engine='openpyxl')
    filas, cols = df.shape
    
    nulos = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    nulos = nulos[nulos > 0]
    
    col_temp, a_min, a_max = extract_temporal_range(df)
    
    print(f'=' * 50)
    print(f'✅ {nombre}')
    print(f'Filas: {filas:,} | Columnas: {cols}')
    if col_temp:
        print(f'Rango Temporal ({col_temp}): {a_min} -> {a_max}')
    else:
        print(f'Rango Temporal: No detectado')
    if not nulos.empty:
        print(f'Top Nulos:\n{nulos.head().to_string()}')
    else:
        print('Nulos: 0%')
    print('\nVista previa:\n')
    display(df.head(3))
    
    resumen.append({
        'Tabla': nombre,
        'Año Inicio': a_min, 
        'Año Fin': a_max
    })

print('=' * 50)


✅ Salud: Consumo SPA
Filas: 97,545 | Columnas: 21
Rango Temporal (ANO): 2015 -> 2025
Top Nulos:
PERTENENCIAETNICA    0.109693
SEXO                 0.041007

Vista previa:



,ANO,SEXO,NOMBRELOCALIDADRESIDENCIA,MESNOTIFICACION,TRIMESTRE,TIPOASEGURAMIENTO,SITIOHABITUALCONSUMO_VIVIENDA,SITIOHABITUALCONSUMO_PARQUE,SITIOHABITUALCONSUMO_EST_EDUCATIVO,SITIOHABITUALCONSUMO_BARES_TABERNAS,...,SITIOHABITUALCONSUMO_CASA_AMIGOS,NIVELEDUCATIVO,CURSO_DE_VIDA,COMOACUDIOTRATAMIENTO,PERTENENCIAETNICA,ORIENTSEXUAL,PAISNACIONALIDAD,NOMBREUPZ,ESTADOCIVIL,CASOS
0,2022,Hombre,Kennedy,6,2,Subsidiado,NO,NO,NO,NO,...,SI,Secundaria completa,Juventud,Voluntariamente,Otros,Heterosexual,Sin Dato,AMERICAS,Soltero (a),1
1,2022,Hombre,Mártires,6,2,Subsidiado,SI,NO,NO,NO,...,NO,Secundaria completa,Adultez,Voluntariamente,Otros,Heterosexual,Sin Dato,LA SABANA,Soltero (a),2
2,2022,Hombre,Kennedy,6,2,Contributivo,NO,SI,SI,NO,...,NO,Secundaria incompleta,Adolescencia,Voluntariamente,Otros,Heterosexual,Sin Dato,CALANDAIMA,Soltero (a),1


✅ Salud: Ideación / Intento
Filas: 254,251 | Columnas: 19
Rango Temporal (ano_notificacion): 2012 -> 2026
Top Nulos:
poblacion_diferencial         7.902820
codigo_localidadresidencia    0.062143

Vista previa:



,ano_notificacion,codigo_localidadresidencia,localidad_residencia,nombre_upz,ciclovital,clasificaciondelaconducta,sexo,edad,niveleducativo,enfermedades_dolorosas,maltrato_sexual,muerte_familiar,conflicto_pareja,problemas_economicos,esc_educ,problemas_juridicos,problemas_laborales,suicidio_amigo,poblacion_diferencial
0,2023,15.0,Antonio Nariño,CIUDAD JARDIN,>60 Vejez,Intento de Suicidio,Hombre,71,3. Primaria incompleta,0,0,0,1,0,0,0,0,0,Ninguna
1,2022,4.0,San Cristóbal,LA GLORIA,>60 Vejez,Intento de Suicidio,Hombre,71,4. Primaria completa,1,0,1,0,0,0,0,0,0,Ninguna
2,2025,19.0,Ciudad Bolívar,LUCERO,>60 Vejez,Ideación suicida,Mujer,71,5. Secundaria incompleta,0,1,0,1,1,0,0,0,0,Ninguna


✅ Salud: Suicidio
Filas: 8,222 | Columnas: 11
Rango Temporal (ANO_DEL_HECHO): 2015 -> 2026
Nulos: 0%

Vista previa:



...

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,ACTIVIDADES PROGRAMAS BOGOTA EN FORMA _I SEMES...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MES,Localidad,Upz,Punto,Direccion,Jornada,# Actividades,ParticipantesHombres,ParticipantesMujeres,Total
2,ENERO,Bosa,El Porvenir,SALON COMUNAL ENTRE PARQUES,Cl. 51 Sur #92a,Manzanas del Ciudado,16,21,98,119


✅ Habitabilidad / Accesibilidad
Filas: 105,493 | Columnas: 15
Rango Temporal (año_indicador): 2011 -> 2024
Top Nulos:
sector_catastral           20.963476
scacodigo                  20.936934
upzcodigo                   6.594750
unidad_planeacion_zonal     6.522708
municipio                   3.408757

Vista previa:



,municipio_codigo,loccodigo,upzcodigo,scacodigo,clase_indicador,tipo_indicador,año_indicador,valor_indicador,categoria_indicador,categoria_secundaria_indicador,nivel_indicador,municipio,localidad,unidad_planeacion_zonal,sector_catastral
0,11001,11.0,NaN,NaN,Condiciones de Habitabilidad y accesibilidad,Estructuras Durables,2016,98.389070,No propiedad horizontal,Sin estrato,Localidad,BOGOTA D.C.,SUBA,NaN,NaN
1,11001,11.0,NaN,NaN,Condiciones de Habitabilidad y accesibilidad,Estructuras Durables,2016,99.973242,Propiedad horizontal,Sin estrato,Localidad,BOGOTA D.C.,SUBA,NaN,NaN
2,11001,11.0,NaN,NaN,Condiciones de Habitabilidad y accesibilidad,Estructuras Durables,2016,96.062992,No propiedad horizontal,Estrato 1,Localidad,BOGOTA D.C.,SUBA,NaN,NaN


✅ Centros Culturales
Filas: 99 | Columnas: 11
Rango Temporal (ano_inicio): 1943 -> 2011
Nulos: 0%

Vista previa:



,nombre_del_museo,telefono_fijo,pagina_web,direccion,localidad,upz,nombre_de_la_entidad_administradora_del_equipamiento,ano_inicio,caracter,del_orden,uso_principal
0,CENTRO MUSICAL LA GAITANA,8976280,http://www.fundacionbatuta.org/,TV 126 133 32,SUBA,TIBABUYES,FUNDACIÓN NACIONAL BATUTA,1996,PÚBLICO,DISTRITAL,Centro Cultural y Artístico
1,ACADEMIA COLOMBIANA DE LA LENGUA,3341190,http://www.academiacolombianadelalengua.co/,AK 3 17 23,SANTA FE,LAS NIEVES,ACADEMIA COLOMBIANA DE LA LENGUA - JAIME POZAD...,1971,PRIVADO,N.D,Centro cultural y artístico
2,PARQUE ZONAL LA AMISTAD,2645100,http://www.idrd.gov.co/sitio/idrd/,KR 78J 41 10 SUR,KENNEDY,KENNEDY CENTRAL,INSTITUTO DISTRITAL DE RECREACIÓN Y DEPORTE -IDRD,1975,PÚBLICO,DISTRITAL,Centro cultural y artístico


In [18]:
df_resumen = pd.DataFrame(resumen)
display(df_resumen)

df_v = df_resumen.dropna()
if not df_v.empty:
    inicio = int(df_v['Año Inicio'].max())
    fin = int(df_v['Año Fin'].min())
    print('\n==== Ventana Temporal Común ====')
    if inicio <= fin:
        print(f'Solapamiento: {inicio} -> {fin}')
    else:
        print('Sin solapamiento completo.')


,Tabla,Año Inicio,Año Fin
0,Salud: Consumo SPA,2015.0,2025.0
1,Salud: Ideación / Intento,2012.0,2026.0
2,Salud: Suicidio,2015.0,2026.0
3,Programas Deporte,NaN,NaN
4,Habitabilidad / Accesibilidad,2011.0,2024.0
5,Centros Culturales,1943.0,2011.0



==== Ventana Temporal Común ====
Sin solapamiento completo.


In [ ]:
import os
import pandas as pd
from IPython.display import display

# Mostramos todo sin truncar las filas
pd.set_option('display.max_rows', None)

print('=== Catálogo Completo de Variables por Base de Datos ===\n')
for nombre, archivo in TABLAS.items():
    ruta = os.path.join(RAW_DIR, archivo)
    if os.path.exists(ruta):
        df = pd.read_excel(ruta, engine='openpyxl')
        tipos = pd.DataFrame({
            'Variable': df.columns,
            'Tipo de Dato': df.dtypes
        })
        print(f'\n🔹 {nombre}')
        display(tipos.reset_index(drop=True))

# Restauramos la opción por defecto por si agregas más celdas luego
pd.reset_option('display.max_rows')
